# 🤖 Deep Q-Network (DQN) Reinforcement Learning
## CartPole — Complete Implementation from Scratch

**Author:** The Architect  
**Framework:** Pure NumPy (no PyTorch/TensorFlow required)  
**Algorithm:** DQN with Double Q-Learning + Experience Replay  
**Environment:** CartPole Inverted Pendulum (built from scratch)

---

### Table of Contents
1. [Setup & Imports](#setup)
2. [Environment Exploration](#env)
3. [Neural Network Architecture](#nn)
4. [DQN Agent Configuration](#agent)
5. [Phase 1: Supervised Pretraining](#pretrain)
6. [Phase 2: RL Fine-tuning](#training)
7. [Results & Visualization](#results)
8. [Model Evaluation](#eval)
9. [Agent Inference Demo](#demo)
10. [Analysis & Conclusions](#conclusions)

## 1. Setup & Imports <a id='setup'></a>

In [ ]:
import sys, os, json, time
import numpy as np

# Add project root to path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Project imports
from model.environment    import CartPoleEnv
from model.neural_network import NeuralNetwork, DenseLayer
from model.replay_buffer  import ReplayBuffer, PrioritisedReplayBuffer
from model.dqn_agent      import DQNAgent, DQNConfig

# Plotting
try:
    import matplotlib
    matplotlib.use('Agg')  # remove this line in interactive Jupyter
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    HAS_MPL = True
    print('✓ matplotlib available')
except ImportError:
    HAS_MPL = False
    print('! matplotlib not available — ASCII fallbacks will be used')

print(f'✓ NumPy {np.__version__}')
print(f'✓ Project root: {PROJECT_ROOT}')
print('✓ All modules loaded')

## 2. Environment Exploration <a id='env'></a>

CartPole is a classic RL benchmark: a cart moves on a frictionless track
and must balance an upright pole by applying left/right forces.

| Component | Value |
|-----------|-------|
| State space | 4-dimensional: [x, ẋ, θ, θ̇] |
| Action space | Binary: {0=left, 1=right} |
| Reward | +1 per step pole is upright |
| Termination | |θ| > 12°, |x| > 2.4, or 500 steps |
| Solve condition | avg reward ≥ 475 over 100 episodes |

In [ ]:
env = CartPoleEnv(seed=42)
obs = env.reset()

print('Initial observation:')
print(f'  cart_position     = {obs[0]:.4f}')
print(f'  cart_velocity     = {obs[1]:.4f}')
print(f'  pole_angle (rad)  = {obs[2]:.4f}')
print(f'  pole_angular_vel  = {obs[3]:.4f}')
print()

# Run random policy for 50 steps
print('Random policy episode:')
obs = env.reset()
total_reward = 0
for step in range(50):
    action = env.sample_action()
    obs, r, term, trunc, info = env.step(action)
    total_reward += r
    if step % 10 == 0:
        print(f'  Step {step:3d}: {env.render()}')
    if term or trunc:
        print(f'  Episode ended at step {step+1}')
        break

print(f'  Total reward (random policy): {total_reward:.0f}')

## 3. Neural Network Architecture <a id='nn'></a>

We implement a fully-connected Q-network from scratch using only NumPy:
- Forward pass
- Backpropagation (chain rule)
- Adam optimiser

Architecture: **4 → 128 → 128 → 2** (input → hidden → hidden → actions)

In [ ]:
net = NeuralNetwork([4, 128, 128, 2], ['relu', 'relu', 'linear'], seed=0)
print(net.summary())

# Test a forward pass
sample_state = np.random.randn(1, 4).astype(np.float32)
q_values = net.predict(sample_state)
print(f'\nSample Q-values: {q_values[0]}')
print(f'Greedy action: {np.argmax(q_values[0])}')

In [ ]:
# Verify backprop with a simple regression task
test_net = NeuralNetwork([4, 32, 32, 2], seed=99)
X_test = np.random.randn(64, 4).astype(np.float32)
Y_test = np.random.randn(64, 2).astype(np.float32)

losses = []
for _ in range(300):
    loss = test_net.train_step(X_test, Y_test, lr=1e-3)
    losses.append(loss)

if HAS_MPL:
    plt.figure(figsize=(8, 3))
    plt.plot(losses)
    plt.title('Neural Network Training Test')
    plt.xlabel('Step'); plt.ylabel('MSE Loss')
    plt.yscale('log'); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/nn_test.png', dpi=100, bbox_inches='tight')
    plt.show()

print(f'Initial loss: {losses[0]:.4f} → Final loss: {losses[-1]:.4f}')
print(f'Loss reduction: {losses[0]/losses[-1]:.1f}x ✓')

## 4. DQN Agent Configuration <a id='agent'></a>

In [ ]:
config = DQNConfig(
    hidden_sizes       = [128, 128],
    learning_rate      = 1e-3,
    batch_size         = 64,
    gamma              = 0.99,
    buffer_capacity    = 50_000,
    min_buffer_size    = 300,
    target_update_freq = 50,
    eps_start          = 0.3,    # lower since we use pretraining
    eps_end            = 0.01,
    eps_decay_steps    = 4000,
    double_dqn         = True,
    seed               = 42,
)
print(config)

## 5. Phase 1: Supervised Pretraining <a id='pretrain'></a>

Instead of starting from a random policy (slow convergence), we use **Behavioural Cloning**:
1. Run an expert heuristic to generate demonstrations
2. Pretrain the Q-network via supervised learning to mimic expert Q-values
3. Use this as a warm-start for RL fine-tuning

The expert policy: push in the direction the pole is leaning.

In [ ]:
def expert_policy(state):
    """Heuristic: follow pole angle with velocity correction."""
    x, xd, theta, td = state
    return 1 if (theta + 0.1*td + 0.05*x) > 0 else 0

def expert_q_values(state):
    """Approximate Q-values for expert action."""
    a = expert_policy(state)
    q = np.array([50.0, 50.0], dtype=np.float32)
    q[a] = 100.0
    return q

# Generate demonstration data
env = CartPoleEnv(seed=42)
X_demo, Y_demo = [], []
for ep in range(30):
    obs = env.reset(seed=42+ep)
    done = False
    count = 0
    while not done and count < 300:
        X_demo.append(obs.copy())
        Y_demo.append(expert_q_values(obs))
        obs, _, term, trunc, _ = env.step(expert_policy(obs))
        done = term or trunc
        count += 1

X_demo = np.array(X_demo, dtype=np.float32)
Y_demo = np.array(Y_demo, dtype=np.float32)
print(f'Demonstration dataset: {len(X_demo):,} state-action pairs')
print(f'  State shape: {X_demo.shape}')
print(f'  Q-target shape: {Y_demo.shape}')

In [ ]:
# Supervised pretraining
pretrain_net = NeuralNetwork([4, 128, 128, 2], ['relu', 'relu', 'linear'], seed=0)
rng = np.random.default_rng(42)
n = len(X_demo)
pretrain_losses = []

N_EPOCHS = 30
for epoch in range(N_EPOCHS):
    idx = rng.permutation(n)
    ep_l = []
    for s in range(0, n, 32):
        b = idx[s:s+32]
        ep_l.append(pretrain_net.train_step(X_demo[b], Y_demo[b], lr=3e-4))
    pretrain_losses.append(float(np.mean(ep_l)))

print(f'Pretraining complete. Loss: {pretrain_losses[0]:.1f} → {pretrain_losses[-1]:.1f}')

if HAS_MPL:
    plt.figure(figsize=(8, 3))
    plt.plot(pretrain_losses, color='#E84C4C', lw=2)
    plt.title('Supervised Pretraining Loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig('results/pretrain_loss.png', dpi=100)
    plt.show()

In [ ]:
# Load the pretrained checkpoint (generated by do_train.py)
pretrained_path = 'checkpoints/pretrained_model.pkl'
if os.path.exists(pretrained_path):
    agent = DQNAgent.load_checkpoint(pretrained_path)
    print('Loaded pretrained checkpoint')
else:
    print('Creating agent from pretrained network...')
    agent = DQNAgent(4, 2, config)
    for sl, dl in zip(pretrain_net.layers, agent.q_net.layers):
        dl.set_params(sl.get_params())
    for sl, dl in zip(pretrain_net.layers, agent.target_net.layers):
        dl.set_params(sl.get_params())

# Evaluate pretrained policy
pretrain_rewards = []
for ei in range(10):
    eobs = env.reset(seed=5000+ei)
    etot = 0.0
    for _ in range(500):
        ea = agent.select_action_greedy(eobs)
        eobs, er, et, etr, _ = env.step(ea)
        etot += er
        if et or etr: break
    pretrain_rewards.append(etot)

print(f'Pretrained policy: mean={np.mean(pretrain_rewards):.1f}  '
      f'max={max(pretrain_rewards):.0f}  '
      f'min={min(pretrain_rewards):.0f}')

## 6. Phase 2: RL Fine-tuning <a id='training'></a>

Load the fully trained model (generated by the training pipeline) or run training here.

In [ ]:
# Load the best trained model
best_model_path = 'checkpoints/best_model.pkl'
final_model_path = 'checkpoints/final_model.pkl'

load_path = best_model_path if os.path.exists(best_model_path) else final_model_path

if os.path.exists(load_path):
    trained_agent = DQNAgent.load_checkpoint(load_path)
    print(f'Loaded model from: {load_path}')
    print(trained_agent.summary())
else:
    print('No checkpoint found — run scripts/do_train.py first')
    trained_agent = None

In [ ]:
# Load training stats
stats_path = 'logs/training_stats.json'
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    print(f'Training stats loaded:')
    print(f'  Total episodes : {len(stats["episode_rewards"])}')
    print(f'  Total steps    : {stats["total_steps"]:,}')
    print(f'  Best eval      : {stats["best_reward"]:.1f}')
    print(f'  Train time     : {stats["elapsed_seconds"]:.1f}s')
else:
    stats = None
    print('No stats file found')

## 7. Results & Visualization <a id='results'></a>

In [ ]:
if stats and HAS_MPL:
    ep_rewards = stats['episode_rewards']
    ep_lengths = stats['episode_lengths']
    losses     = stats['losses']
    epsilons   = stats['epsilons']
    eval_log   = stats['eval_log']

    def ma(x, w=20):
        x = np.array(x, dtype=float)
        if len(x) < w: return x
        pad = np.full(w-1, x[0])
        return np.convolve(np.concatenate([pad, x]), np.ones(w)/w, 'valid')

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle('DQN Training Results — CartPole', fontsize=14, fontweight='bold')

    # Episode rewards
    ax = axes[0, 0]
    ax.plot(ep_rewards, alpha=0.3, color='#4C9BE8', lw=0.8, label='Episode')
    ax.plot(range(len(ma(ep_rewards))), ma(ep_rewards), color='#1A5FAD', lw=2, label='MA-20')
    ax.axhline(475, color='green', ls='--', alpha=0.7, label='Solve')
    ax.set_title('Episode Rewards'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Loss
    ax = axes[0, 1]
    if losses:
        ax.plot(losses, alpha=0.2, color='#E84C4C', lw=0.5)
        ax.plot(range(len(ma(losses, 100))), ma(losses, 100), color='#A01010', lw=2)
        ax.set_title('Training Loss (MSE)'); ax.set_xlabel('Step')
        ax.set_yscale('log'); ax.grid(True, alpha=0.3)

    # Epsilon
    ax = axes[1, 0]
    if epsilons:
        ax.plot(epsilons, color='#E8A14C', lw=1.5)
        ax.set_title('Epsilon Decay'); ax.set_xlabel('Step')
        ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

    # Eval
    ax = axes[1, 1]
    if eval_log:
        ep_x   = [e['episode'] for e in eval_log]
        mean_r = [e['mean_reward'] for e in eval_log]
        ax.plot(ep_x, mean_r, color='#4CAF50', lw=2, marker='o', ms=5)
        ax.axhline(475, color='green', ls='--', alpha=0.7, label='Solve')
        ax.set_title('Eval Reward (Greedy)'); ax.set_xlabel('Episode')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/notebook_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✓ Training curves plotted')
elif stats:
    ep_rewards = stats['episode_rewards']
    print(f'Mean last 50 episodes: {np.mean(ep_rewards[-50:]):.1f}')
    print(f'Max episode reward: {max(ep_rewards):.0f}')

## 8. Model Evaluation <a id='eval'></a>

In [ ]:
from evaluate import evaluate

if os.path.exists('checkpoints/best_model.pkl'):
    print('=== Evaluating Best Model ===')
    results = evaluate('checkpoints/best_model.pkl', n_episodes=15, seed=2000)
    print(f'\nResults: {results}')

## 9. Agent Inference Demo <a id='demo'></a>

In [ ]:
if trained_agent:
    from evaluate import render_episode
    print('=== One Episode Rendered ===')
    render_episode('checkpoints/best_model.pkl', seed=42)

In [ ]:
# Q-value inspection
if trained_agent:
    print('Q-value Analysis')
    print('='*50)
    test_states = [
        np.array([0.0, 0.0,  0.0,  0.0], dtype=np.float32),  # upright balanced
        np.array([0.0, 0.0,  0.1,  0.0], dtype=np.float32),  # leaning right
        np.array([0.0, 0.0, -0.1,  0.0], dtype=np.float32),  # leaning left
        np.array([1.0, 0.5,  0.05, 0.1], dtype=np.float32),  # moving right
    ]
    labels = ['Balanced', 'Lean right', 'Lean left', 'Moving right']

    for state, label in zip(test_states, labels):
        q_vals = trained_agent.q_net.predict(state.reshape(1,-1))[0]
        action = np.argmax(q_vals)
        action_str = 'LEFT' if action == 0 else 'RIGHT'
        print(f'  {label:15s}: Q=[{q_vals[0]:7.2f}, {q_vals[1]:7.2f}]  → {action_str}')

## 10. Analysis & Conclusions <a id='conclusions'></a>

In [ ]:
print('=' * 60)
print('  DQN Training Summary')
print('=' * 60)

if stats:
    ep_rewards = stats['episode_rewards']
    eval_log   = stats['eval_log']
    
    n_ep = len(ep_rewards)
    print(f'  Total episodes    : {n_ep}')
    print(f'  Total env steps   : {stats["total_steps"]:,}')
    print(f'  Train time        : {stats["elapsed_seconds"]:.1f}s')
    print(f'  Best eval reward  : {stats["best_reward"]:.1f}')
    print(f'  Final avg-100     : {np.mean(ep_rewards[-100:]):.1f}')
    
    if eval_log:
        best_ep = max(eval_log, key=lambda e: e['mean_reward'])
        print(f'  Best eval episode : {best_ep["episode"]} (mean={best_ep["mean_reward"]:.1f})')

print()
print('  Key Techniques Used:')
print('  ✓ Neural Q-network (NumPy, no PyTorch)')
print('  ✓ Experience Replay Buffer')
print('  ✓ Target Network (hard update)')
print('  ✓ Double DQN (reduces overestimation bias)')
print('  ✓ ε-greedy exploration with decay')
print('  ✓ Supervised Pretraining (behavioural cloning)')
print('  ✓ Adam optimiser')
print('  ✓ Gradient clipping')
print('=' * 60)